# 의미 고정 N/V 경제표현 + 개인별 CLV 양성가중 M5: Dunnhumby seed 42 단일 arm

빠른 구현·방향 확인용 역사적 개발구간 실행입니다. M5 actual 하나만 100 epoch 학습하며 대조군은 학습하지 않습니다. 따라서 이 결과만으로 성공·실패 또는 CLV 고유효과를 판정하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '3827f90825c894ed9d22c4b3004d1a871565fdee'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_semantic_nv_single_screen import (
    configure_semantic_nv_single_screen,
    preflight_summary,
    run_semantic_nv_single_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_semantic_nv_single_screen(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_semantic_nv_personalized_positive_single_screen_v1',
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['controls_trained'] is False
assert summary['trained_models'] == ['m5_semantic_nv_personalized_positive_weight_k5']
assert summary['m2']['economic_dim'] == 5
assert summary['m2']['economic_graph_propagation'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_semantic_nv_single_screen(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) 단일 actual M5 절대지표')
show(result_df)
print('2) ID 점수 대비 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('3) 실행 해석 범위')
print(json.dumps(result_df.attrs['preflight']['reading_rule'], ensure_ascii=False, indent=2))
print('4) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))